In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

#
def _parse_function(proto):
  
    keys_to_features = {
        'img_gt': tf.io.FixedLenFeature([], tf.string),
        'map_gt': tf.io.FixedLenFeature([], tf.string),
        'seg': tf.io.FixedLenFeature([], tf.string),
        'rician': tf.io.FixedLenFeature([], tf.string),
        'tes': tf.io.FixedLenFeature([], tf.string),
        'Nx': tf.io.FixedLenFeature([], tf.int64),
        'Ny': tf.io.FixedLenFeature([], tf.int64),
        'Nc': tf.io.FixedLenFeature([], tf.int64),
        'Np': tf.io.FixedLenFeature([], tf.int64),
    }


    parsed_features = tf.io.parse_single_example(proto, keys_to_features)

    
    img_gt = tf.io.parse_tensor(parsed_features['img_gt'], out_type=tf.float32)
    map_gt = tf.io.parse_tensor(parsed_features['map_gt'], out_type=tf.float32)
    seg = tf.io.parse_tensor(parsed_features['seg'], out_type=tf.float32)
    rician = tf.io.parse_tensor(parsed_features['rician'], out_type=tf.float32)
    tes = tf.io.parse_tensor(parsed_features['tes'], out_type=tf.float32)

    return img_gt, map_gt, seg, rician, tes






def visualize_tfrecord(tfrecord_path, sample_index=None):
 
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    dataset = dataset.map(_parse_function)

    
    for idx, (img_gt, map_gt, seg, rician, tes) in enumerate(dataset):
        if sample_index is not None and idx != sample_index:
            continue

        # 
        Nx, Ny, Nc = img_gt.shape[1], img_gt.shape[2], img_gt.shape[3]
        Np = map_gt.shape[-1] if len(map_gt.shape) == 4 else 1

        # 
        for echo_idx in range(img_gt.shape[0]):
            print(f"Sample: {idx}, Echo: {echo_idx}, TE: {tes[echo_idx].numpy()}")

            # 
            plt.figure(figsize=(18, 4))

            plt.subplot(1, 5, 1)
            plt.title(f"Echo {echo_idx} Image")
            plt.imshow(img_gt[echo_idx, :, :, 0], cmap='gray')
            plt.colorbar()

            plt.subplot(1, 5, 2)
            plt.title(f"Echo {echo_idx} Map Channel 0")
            if Np == 2:
                plt.imshow(map_gt[echo_idx, :, :, 0], cmap='jet')
            else:
                plt.imshow(map_gt[echo_idx, :, :], cmap='jet')
            plt.colorbar()

            plt.subplot(1, 5, 3)
            plt.title(f"Echo {echo_idx} Map Channel 1")
            if Np == 2:
                plt.imshow(map_gt[echo_idx, :, :, 1], cmap='jet')
            plt.colorbar()

            plt.subplot(1, 5, 4)
            plt.title(f"Echo {echo_idx} Mask")
            plt.imshow(seg[echo_idx, :, :, 0], cmap='gray')
            plt.colorbar()

            plt.subplot(1, 5, 5)
            plt.title(f"Echo {echo_idx} Rician")
            plt.imshow(rician[echo_idx, :, :, 0], cmap='gray')
            plt.colorbar()

            plt.show()

        if sample_index is not None:
            break

#
def main():
    base_path = '/root/autodl-tmp/data/train'
    tfrecord_path = os.path.join(base_path, 'traindata.tfrecords')

    # 
    sample_index_to_visualize = 57
    visualize_tfrecord(tfrecord_path, sample_index=sample_index_to_visualize)

# 执行入口
if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
from collections import OrderedDict

def extract_te_from_dicom(file_path: str) -> float:
   
    try:
        #
        import pydicom
        ds = pydicom.dcmread(file_path)
        if hasattr(ds, 'EchoTime'):
            return float(ds.EchoTime)
        elif hasattr(ds, 'EchoTimes') and len(ds.EchoTimes) > 0:
            return float(ds.EchoTimes[0])
        return None
    except ImportError:
        print("pydicom未安装，使用二进制方法")
        return extract_te_binary(file_path)
    except Exception as e:
        print(f"读取DICOM文件出错: {e}")
        return extract_te_binary(file_path)

def extract_te_binary(file_path: str) -> float:
    
    try:
        with open(file_path, 'rb') as f:
            content = f.read()
      
        te_patterns = [
            b'\x18\x00\x81\x00',  # 
            b'\x00\x18\x00\x81'   # 
        ]
        
        for pattern in te_patterns:
            pos = content.find(pattern)
            if pos != -1 and pos + 8 < len(content):
              
                te_bytes = content[pos+4:pos+8]
                te_value = np.frombuffer(te_bytes, dtype=np.float32)[0]
                return float(te_value)
        
        return None
    except:
        return None

def scan_and_group_te_patterns(root_dir: str, echo_cnt: int = 12):
 
    te_patterns = OrderedDict()
    sample_count = 0
    
    if not os.path.exists(root_dir):
        print(f"错误: 路径不存在: {root_dir}")
        return te_patterns
    
    for dirpath, _, filenames in os.walk(root_dir):
        dcm_files = sorted([f for f in filenames if f.lower().endswith('.dcm')])
        
        if len(dcm_files) != echo_cnt:
            continue
            
        full_paths = [os.path.join(dirpath, f) for f in dcm_files]
        tes = []
        
        for file_path in full_paths:
            te = extract_te_from_dicom(file_path)
            if te is None:
                break
            tes.append(te)
        
        if len(tes) != echo_cnt:
            continue
            
        #
        te_pattern_str = ','.join([f"{te:.2f}" for te in tes])
        
        if te_pattern_str not in te_patterns:
            te_patterns[te_pattern_str] = {
                'pattern': tes.copy(),
                'count': 0,
                'example_dir': os.path.basename(dirpath)
            }
        
        te_patterns[te_pattern_str]['count'] += 1
        sample_count += 1
    
    return te_patterns, sample_count

# -------------------------------------
if __name__ == '__main__':
    root = '/root/autodl-tmp/data/train/dcm'
    
    print("开始扫描DICOM文件并分组TE模式...")
    te_patterns, total_samples = scan_and_group_te_patterns(root, echo_cnt=12)
    
    print(f"\n共扫描 {total_samples} 个12回波样本")
    print(f"发现 {len(te_patterns)} 种不同的TE模式:\n")
    
    for i, (pattern_str, info) in enumerate(te_patterns.items(), 1):
        print(f"模式 {i}:")
        print(f"  TE序列: {np.round(info['pattern'], 4)}")
        print(f"  出现次数: {info['count']}")
        print(f"  示例目录: {info['example_dir']}")
        print()
    
    # 
    output_file = "te_patterns.txt"
    with open(output_file, 'w') as f:
        f.write("所有不同的回波时间模式:\n")
        f.write("=" * 50 + "\n\n")
        
        for i, (pattern_str, info) in enumerate(te_patterns.items(), 1):
            f.write(f"模式 {i}:\n")
            f.write(f"TE序列: {np.round(info['pattern'], 4)}\n")
            f.write(f"出现次数: {info['count']}\n")
            f.write(f"示例目录: {info['example_dir']}\n")
            f.write("-" * 30 + "\n")
    
    print(f"TE模式已保存到: {output_file}")
    
    
    all_unique_patterns = [info['pattern'] for info in te_patterns.values()]
    print(f"\n提取到 {len(all_unique_patterns)} 种不同的TE模式数组")

In [ ]:
import os
import numpy as np

def check_last_dim_not_2(directory):
   
    if not os.path.isdir(directory):
        print(f"错误：路径 '{directory}' 不是一个有效的目录")
        return

    print(f"开始检查目录 '{directory}' 下的 .npy 文件...\n")

    for filename in os.listdir(directory):
        if filename.endswith(".npy"):
            file_path = os.path.join(directory, filename)
            try:
                data = np.load(file_path)

                # 
                if len(data.shape) == 0:
                    print(f"{file_path} 是 0D 数据（标量），跳过。")
                    continue

                last_dim = data.shape[-1]

                if last_dim != 1:
                    print(f"❌ 文件: {file_path}")
                    print(f"   形状: {data.shape}\n")

            except Exception as e:
                print(f"加载文件 {file_path} 出错: {e}\n")


#
if __name__ == "__main__":
    data_dir = "/root/autodl-tmp/data/train/mask"  # 
    check_last_dim_not_2(data_dir)

In [ ]:
import tensorflow as tf
import numpy as np
import os

def parse_example(example_proto):
    feature_description = {
        'img_gt': tf.io.FixedLenFeature([], tf.string),
        'map_gt': tf.io.FixedLenFeature([], tf.string),
        'seg': tf.io.FixedLenFeature([], tf.string),
        'rician': tf.io.FixedLenFeature([], tf.string),
        'tes': tf.io.FixedLenFeature([], tf.string),
        'Nx': tf.io.FixedLenFeature([], tf.int64),
        'Ny': tf.io.FixedLenFeature([], tf.int64),
        'Nc': tf.io.FixedLenFeature([], tf.int64),
        'Np': tf.io.FixedLenFeature([], tf.int64),
    }
    parsed_features = tf.io.parse_single_example(example_proto, feature_description)

    # 
    img_gt = tf.io.parse_tensor(parsed_features['img_gt'], out_type=tf.float32)
    map_gt = tf.io.parse_tensor(parsed_features['map_gt'], out_type=tf.float32)
    seg = tf.io.parse_tensor(parsed_features['seg'], out_type=tf.float32)
    rician = tf.io.parse_tensor(parsed_features['rician'], out_type=tf.float32)
    tes = tf.io.parse_tensor(parsed_features['tes'], out_type=tf.float32)

    return img_gt, map_gt, seg, rician, tes

def inspect_tfrecord_shapes(tfrecord_path):
    if os.path.isdir(tfrecord_path):
        tfrecord_files = [os.path.join(tfrecord_path, f) for f in os.listdir(tfrecord_path) if f.endswith('.tfrecords')]
    else:
        tfrecord_files = [tfrecord_path]

    unique_img_shapes = set()
    unique_map_shapes = set()
    unique_seg_shapes =set()
    unique_tes_shapes = set()

    for file in tfrecord_files:
        print(f"🔍 Processing file: {file}")
        dataset = tf.data.TFRecordDataset(file)

        for raw_record in dataset:
            img, map_, seg, rician, tes = parse_example(raw_record)

            # 
            img_shape = tuple(img.numpy().shape)
            map_shape = tuple(map_.numpy().shape)
            seg_shape = tuple(seg.numpy().shape)
            tes_shape = tuple(tes.numpy().shape)

            # 
            unique_img_shapes.add(img_shape)
            unique_map_shapes.add(map_shape)
            unique_seg_shapes.add(seg_shape)
            unique_tes_shapes.add(tes_shape)

    print("\n📊 Unique shapes found:")
    print("----------- img_gt -----------")
    for shape in sorted(unique_img_shapes):
        print(shape)

    print("----------- map_gt -----------")
    for shape in sorted(unique_map_shapes):
        print(shape)
    print("----------- seg_gt -----------")
    for shape in sorted(unique_seg_shapes):
        print(shape)


    print("----------- tes -----------")
    for shape in sorted(unique_tes_shapes):
        print(shape)

# ---------------- 主程序入口 ----------------
if __name__ == "__main__":
    tfrecord_dir = "/root/autodl-tmp/data/train/"  # 
    inspect_tfrecord_shapes(tfrecord_dir)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import numpy as np
from pathlib import Path

# 
ROOT_DIR ='/root/autodl-tmp/data/train/map_r2'      #


def main(root_dir):
    root = Path(root_dir).resolve()
    print(f'扫描目录: {root}\n')
    npy_files = list(root.rglob('*.npy'))
    if not npy_files:
        print('未找到任何 .npy 文件！')
        return
    for npy in npy_files:
        try:
            arr = np.load(npy, mmap_mode='r')
            print(f'{npy.relative_to(root)}  ->  shape = {arr.shape}')
        except Exception as e:
            print(f'{npy.relative_to(root)}  ->  加载失败: {e}')

if __name__ == '__main__':
    main(ROOT_DIR)